# Phase 3.6 — teaching the model to say no

`google/flan-t5-small`, trained to answer from retrieved law. Resumable: if Colab
disconnects, run all again and it picks up from the last finished epoch on Drive.

## How to run it

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**, and allow Google Drive access when asked
3. Training takes roughly **2 hours**. A disconnect costs at most one epoch.

## Where this sits

**Phase 3.5 worked, on the test set.** Generating the training context with the
bot's own assembly code — four passages, not one — took citation accuracy on the
676 held-out questions from 86.8% to 92.7%, the old-to-new correspondence to
99.1% and the new-to-old to 100%. Retrieval is now within 0.5 F1 of its own
oracle, so there is little left to win there.

**It did nothing for the confusion set**, which stayed at 23.3% in every
condition, with exact citation accuracy at zero. Counting the answer shapes showed
why, and it was not the model's size:

| | Answers opening with a negation |
|---|---|
| Training set | 41 / 11,751 — **0.3%** |
| Confusion set | 43 / 44 — **97.7%** |

The model had seen "X corresponds to Y" eleven thousand times and almost never a
question whose correct answer is that the correspondence does not hold. So it
always produced a correspondence — inventing a "BNS Section 105L" for sedition,
which does not exist. Worse, 33 of the 44 confusion questions are `collision`,
`merged` or `split` shapes, and the training data contained **no question of those
shapes at all**. It was not a harder version of a trained task; it was an
untrained one.

## What is new in this run

`scripts/build_negation_dataset.py` adds 1,006 rows of the missing shapes, built
from the same concordance for provisions outside every held-out group, which lifts
negations in the training mix from 0.3% to 6.4%:

| Shape | Question | Rows |
|---|---|---|
| `collision` | Does BNSS 298 deal with the same subject as CrPC 298? | 700 |
| `merged` | Is voluntarily causing hurt still dealt with under IPC 323? | 212 |
| `no_single` | Which single IPC section corresponds to BNS 126? | 94 |

Two honest caveats, both recorded in `RESULTS.md`. All five split families in the
concordance sit inside held-out groups, so no `split` example could be generated
without leaking; `no_single` teaches the same answer shape from many-to-one
families instead, and five confusion questions stay shape-novel. And the confusion
set's question *templates* now appear in training on different provisions — it
remains held out at the level of fact, but not of phrasing.

Everything else is unchanged: same splits, same held-out sets, still selecting on
citation accuracy rather than validation loss.

In [ ]:
# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/suryanshu-g/legal-llm-bot.git"
HF_REPO_ID = "legal-llm-bot-flan-t5-small-context-v3"

SEED = 20240701
MODEL_NAME = "google/flan-t5-small"

# Everything that must survive a disconnect lives here, on Google Drive.
DRIVE_DIR = "/content/drive/MyDrive/legal-llm-bot"

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "No GPU detected - set Runtime > Change runtime type > T4 GPU")

In [ ]:
# faiss and sentence-transformers are needed here but not in Phase 2: this
# notebook runs live retrieval over the test set at evaluation time.
%pip install -q -U transformers datasets accelerate sentencepiece sentence-transformers faiss-cpu

> **If Colab shows a "RESTART SESSION" button after the install, click it**, then
> carry on from the next cell. You do not need to re-run the install.

In [ ]:
import importlib, platform
print("python      ", platform.python_version())
for mod in ("torch", "transformers", "datasets", "accelerate", "numpy",
            "sentence_transformers", "faiss"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<22} {getattr(m, '__version__', 'ok')}")
    except Exception as exc:
        print(f"{mod:<22} NOT IMPORTABLE: {exc}")

In [ ]:
import os, json, re, time, random, subprocess, textwrap
from collections import Counter, defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

import transformers
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, EarlyStoppingCallback,
                          set_seed)
from datasets import Dataset

# The Trainer API renames arguments between major versions and Colab tracks the
# latest release, so nothing here is pinned - the notebook asks the installed
# classes what they accept.
import inspect

TRAINER_ARGS_ACCEPTS = set(
    inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
TRAINER_ACCEPTS = set(inspect.signature(Seq2SeqTrainer.__init__).parameters)
TOKENIZER_KEY = ("processing_class" if "processing_class" in TRAINER_ACCEPTS
                 else "tokenizer")

# Where a setting has been renamed, list the candidates in preference order.
# transformers 5 dropped warmup_ratio; its warmup_steps accepts a float below 1
# and means the same thing, but in v4 warmup_steps is an integer step count, so
# warmup_ratio has to be tried first.
ARG_ALIASES = {
    "eval_strategy": ("eval_strategy", "evaluation_strategy"),
    "warmup_ratio": ("warmup_ratio", "warmup_steps"),
}


def make_training_args(**desired):
    """Build Seq2SeqTrainingArguments using the names this version accepts."""
    kwargs, notes = {}, []
    for key, value in desired.items():
        for name in ARG_ALIASES.get(key, (key,)):
            if name in TRAINER_ARGS_ACCEPTS:
                kwargs[name] = value
                if name != key:
                    notes.append(f"{key} -> {name}")
                break
        else:
            notes.append(f"{key} DROPPED (not in transformers "
                         f"{transformers.__version__})")
    return Seq2SeqTrainingArguments(**kwargs), notes


print(f"transformers {transformers.__version__} | "
      f"Trainer takes '{TOKENIZER_KEY}'")

set_seed(SEED)
random.seed(SEED); np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| device", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
REPO_DIR = "/content/legal-llm-bot"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("already cloned")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed - check REPO_URL: {REPO_URL}")

DATA = os.path.join(REPO_DIR, "data", "processed")
sys.path.insert(0, os.path.join(REPO_DIR, "scripts"))   # for retrieve.py

missing = [f for f in ("train_context.jsonl", "val_context.jsonl",
                       "test.jsonl", "test_index.jsonl",
                       "confusion_test_set.jsonl", "retrieval_index.faiss",
                       "retrieval_index_meta.jsonl")
           if not os.path.exists(os.path.join(DATA, f))]
if missing:
    raise FileNotFoundError(f"missing data files: {missing}")
print("data ready:", len(os.listdir(DATA)), "files")

In [ ]:
# Mount Drive now, because training checkpoints are written there. A disconnect
# then costs at most the epoch in progress: re-open the notebook, Run all, and
# training resumes from the last saved epoch.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("progress will be saved under", DRIVE_DIR)

---

## Data

`train_context.jsonl` and `val_context.jsonl` carry the same
`instruction` / `input` / `output` schema as Phase 2, with the retrieved passage
in the `input` slot — the slot Phase 2's prompt template already reserved, so
the input format the model sees is unchanged. Two extra fields, `context_type`
and `qa_type`, exist purely so evaluation can be broken down by condition.

**The test and confusion sets are deliberately not pre-baked with context.**
Their context comes from live retrieval further down, which means the test
number measures the whole pipeline — retriever included, with its real error
rate — rather than a best case that assumes retrieval always works.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

ctx = {n: load_jsonl(os.path.join(DATA, f"{n}_context.jsonl"))
       for n in ("train", "val")}
test_rows = load_jsonl(os.path.join(DATA, "test.jsonl"))
test_index = load_jsonl(os.path.join(DATA, "test_index.jsonl"))
confusion = load_jsonl(os.path.join(DATA, "confusion_test_set.jsonl"))

for name, rows in ctx.items():
    print(f"{name:<6} {len(rows):>6} rows  "
          f"{dict(Counter(r['context_type'] for r in rows))}")
print(f"{'test':<6} {len(test_rows):>6} rows  (context added live, below)")
print(f"{'conf':<6} {len(confusion):>6} rows  (held out of every split)")

In [ ]:
# Condition against question type, so nothing is concentrated in one corner.
types = sorted({r["qa_type"] for r in ctx["train"]})
conds = ("positive", "distractor", "none")
print(f"{'qa_type':<24}" + "".join(f"{c:>12}" for c in conds) + f"{'total':>8}")
for t in types:
    sub = [r for r in ctx["train"] if r["qa_type"] == t]
    counts = Counter(r["context_type"] for r in sub)
    print(f"{t:<24}" + "".join(f"{counts[c]:>12}" for c in conds)
          + f"{len(sub):>8}")

In [ ]:
# One example of each condition. The distractor is the one to read: the context
# is plausible and wrong, and the target answer ignores it.
for cond in conds:
    row = next(r for r in ctx["train"] if r["context_type"] == cond)
    print("=" * 78)
    print(f"[{cond}]  chunk: {row['context_chunk_id'] or '(none)'}"
          f"   qa_type: {row['qa_type']}")
    print("  Q  :", textwrap.shorten(row["instruction"], 92, placeholder=" ..."))
    print("  CTX:", textwrap.shorten(row["context"] or "(empty)", 300,
                                     placeholder=" ..."))
    print("  A  :", textwrap.shorten(row["output"], 160, placeholder=" ..."))

---

## Preprocessing

Same prompt template as Phase 2 — deliberately, because that is what makes the
two runs comparable:

```
answer the indian criminal law question: {instruction}

context: {retrieved passage}
```

The only change is the encoder window. Phase 2 needed 96 tokens because the
input was a bare question; with a passage attached the measured prompt length is
median 163, p99 475, max 495 tokens, so **512** truncates nothing. Passages were
already capped at 448 tokens when the dataset was built.

`flan-t5-small` is light enough to take a batch of 16 directly at this length,
so the effective batch matches Phase 2 without gradient accumulation.

In [ ]:
TASK_PREFIX = "answer the indian criminal law question: "
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 320

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def build_prompt(row):
    """The input string the model sees. Phase 3 fills `input` with retrieval."""
    prompt = TASK_PREFIX + row["instruction"].strip()
    context = (row.get("input") or "").strip()
    if context:
        prompt += "\n\ncontext: " + context
    return prompt

# Overridden for context: measured max prompt is 495 tokens.
MAX_SOURCE_LENGTH = 512
print('source', MAX_SOURCE_LENGTH, '| target', MAX_TARGET_LENGTH)
print(build_prompt(ctx['train'][0])[:300])

In [ ]:
# Confirm the window before committing to it.
for name, rows in ctx.items():
    src = [len(tokenizer(build_prompt(r)).input_ids) for r in rows]
    tgt = [len(tokenizer(r["output"]).input_ids) for r in rows]
    print(f"{name:<6} source max {max(src):>4} (> {MAX_SOURCE_LENGTH}: "
          f"{sum(1 for x in src if x > MAX_SOURCE_LENGTH)})"
          f"   target max {max(tgt):>4} (> {MAX_TARGET_LENGTH}: "
          f"{sum(1 for x in tgt if x > MAX_TARGET_LENGTH)})")

In [ ]:
def tokenize(batch):
    model_inputs = tokenizer(batch["prompt"], max_length=MAX_SOURCE_LENGTH,
                             truncation=True)
    labels = tokenizer(text_target=batch["output"], max_length=MAX_TARGET_LENGTH,
                       truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def to_dataset(rows):
    ds = Dataset.from_list([{"prompt": build_prompt(r), "output": r["output"]}
                            for r in rows])
    return ds.map(tokenize, batched=True, remove_columns=ds.column_names,
                  desc="tokenizing")


tokenized = {n: to_dataset(rows) for n, rows in ctx.items()}
print(tokenized["train"])

---

## Stopping criterion

Phase 2 stopped on validation loss, and produced a model that had learnt the
answer format and almost none of the facts. Across 214 held-out questions asking
what a provision became after 1 July 2024 it was right **zero** times, while
token F1 read 58%.

Validation loss is a poor proxy on this data. The answers are heavily
templated — most of the token mass is boilerplate like "The Bharatiya Nyaya
Sanhita, 2023 replaced the Indian Penal Code, 1860 with effect from 1 July
2024" — so cross-entropy flattens once the template is learnt, while the one or
two tokens carrying the section number are still wrong. The run looks converged
and is factually useless.

This run therefore selects on **citation accuracy**: the fraction of validation
answers whose statutory citations exactly match the gold answer's. That is the
capability the project exists to provide, and it cannot be gamed by fluent
boilerplate.

Generating during training costs time, so the in-training metric is computed on
a fixed 150-row stratified slice of validation rather than all 1,369, and
answers are generated only far enough (128 tokens) to reach the citations,
which come in the first sentence. This slice is the main thing that keeps each
epoch short enough for a free Colab session.

In [ ]:
# The metrics live in scripts/metrics.py so that this notebook and Phase 4's
# score_phase4.py cannot drift apart - comparing this bot against another model
# is meaningless if the two are scored by different code.
from metrics import (ACT_ALIASES, exact_match, extract_refs, normalise, pct,
                     score, summarise, token_f1)

print("metrics imported from scripts/metrics.py")

In [ ]:
# metrics.py self-checks its reference extractor on import, against the citation
# forms that actually occur - including subsection forms like "Section 115(2) of
# the Bharatiya Nyaya Sanhita", which an earlier version missed entirely.
print("reference extractor: checks passed at import")

# Why the citation metric earns its place: token F1 hardly notices a wrong section.
_gold = "IPC Section 302 corresponds to BNS Section 103 (Punishment for murder)."
_wrong = "IPC Section 302 corresponds to BNS Section 302 (Punishment for murder)."
print(f"  a wrong-section answer scores token F1 {token_f1(_wrong, _gold):.3f} "
      f"but fails the citation check "
      f"({extract_refs(_gold) <= extract_refs(_wrong)})")

In [ ]:
# A stratified validation slice for the in-training metric: all three context
# conditions represented, fixed by seed so every epoch is scored on the same
# rows.
rng = random.Random(SEED)
val_sub_rows = []
for cond in conds:
    pool = [r for r in ctx["val"] if r["context_type"] == cond]
    take = max(1, round(150 * len(pool) / len(ctx["val"])))
    val_sub_rows += rng.sample(pool, min(take, len(pool)))
print("in-training eval slice:",
      dict(Counter(r["context_type"] for r in val_sub_rows)),
      f"= {len(val_sub_rows)} rows")

val_sub = to_dataset(val_sub_rows)
gold_texts = [r["output"] for r in val_sub_rows]


def compute_metrics(eval_pred):
    """Citation accuracy over generated answers - drives early stopping."""
    preds = eval_pred.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    decoded = tokenizer.batch_decode(preds, skip_special_tokens=True)

    exact = subset = f1 = scored = 0
    for pred, gold in zip(decoded, gold_texts):
        g, p = extract_refs(gold), extract_refs(pred)
        f1 += token_f1(pred, gold)
        if not g:
            continue
        scored += 1
        exact += g == p
        subset += g <= p
    return {
        "citation_exact": 100 * exact / max(scored, 1),
        "citation_subset": 100 * subset / max(scored, 1),
        "token_f1": 100 * f1 / max(len(decoded), 1),
    }

---

## Model and training

`google/flan-t5-small` (77M parameters), learning rate 3e-4 with linear decay
and 5% warmup, AdamW with 0.01 weight decay, fp32 because T5 overflows in fp16
and the T4 has no bf16, and the best checkpoint by citation accuracy kept rather
than the last.

A 10-epoch ceiling with early stopping at patience 2, rather than Phase 2's
25 and 3: at this size the model settles quickly, and the tighter limits keep
the whole run inside a free Colab session.

**Resuming.** Checkpoints are written to Drive each epoch. The training cell
checks for them first: if training already finished in an earlier session it
loads the finished model and moves on; if it was interrupted part-way it resumes
from the last saved epoch.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"{model.num_parameters() / 1e6:.1f}M parameters")

# bf16 only where the hardware supports it; never fp16 for T5 (NaN losses).
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("bf16:", USE_BF16, "| fp16: False (T5 overflows in fp16)")

In [ ]:
# On Drive, so a disconnect does not throw away finished epochs.
OUTPUT_DIR = os.path.join(DRIVE_DIR, "checkpoints-flan-t5-small-context-v3")
FINAL_DIR = os.path.join(DRIVE_DIR, "flan-t5-small-context-v3")
DONE_MARKER = os.path.join(FINAL_DIR, "TRAINING_DONE.json")

# flan-t5-small fits a batch of 16 at 512 tokens; effective batch matches Phase 2.
BATCH_SIZE = 16
GRAD_ACCUM = 1

args, notes = make_training_args(
    output_dir=OUTPUT_DIR,
    seed=SEED,
    num_train_epochs=10,                 # ceiling; early stopping decides
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_torch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=GRAD_ACCUM,
    bf16=USE_BF16,
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    # Phase 2 selected on eval_loss and stopped on a model that was fluent and
    # factually wrong. Citation accuracy cannot be reached by boilerplate.
    metric_for_best_model="citation_exact",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=128,           # citations come in the first sentence
    report_to="none",
)
for note in notes:
    print("  adapted:", note)

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=val_sub,                # the stratified slice, for speed
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    **{TOKENIZER_KEY: tokenizer},
)
print("effective batch size:", BATCH_SIZE * GRAD_ACCUM)
print("selecting on:", args.metric_for_best_model)

In [ ]:
# ---- train, resume, or skip ----------------------------------------------
if os.path.exists(DONE_MARKER):
    # Finished in an earlier session: load the saved model, skip training.
    print("Training already finished in an earlier session.")
    print("Loading the saved model from", FINAL_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(DEVICE)
    with open(DONE_MARKER, encoding="utf-8") as fh:
        done = json.load(fh)
    HISTORY = done["log_history"]
    TRAIN_MINUTES = done["training_minutes"]
else:
    checkpoints = ([d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
                   if os.path.isdir(OUTPUT_DIR) else [])
    if checkpoints:
        print(f"Resuming from the last saved epoch ({len(checkpoints)} "
              f"checkpoint(s) found on Drive).")
    else:
        print("Starting training from scratch.")

    t0 = time.time()
    trainer.train(resume_from_checkpoint=True if checkpoints else None)
    TRAIN_MINUTES = (time.time() - t0) / 60

    # load_best_model_at_end has put the best epoch's weights on the trainer.
    model = trainer.model
    trainer.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)
    HISTORY = trainer.state.log_history
    with open(DONE_MARKER, "w", encoding="utf-8") as fh:
        json.dump({"training_minutes": round(TRAIN_MINUTES, 1),
                   "log_history": HISTORY}, fh, indent=2)
    print(f"\nTraining finished in {TRAIN_MINUTES:.1f} min (this session).")
    print("Best model saved to", FINAL_DIR)

model.eval()

In [ ]:
# Loss, and the citation accuracy that actually decided when to stop. The
# Phase 2 failure looked like this: loss improving while citations stayed at 0.
tr = [(h["epoch"], h["loss"]) for h in HISTORY if "loss" in h]
ev = [(h["epoch"], h["eval_loss"]) for h in HISTORY if "eval_loss" in h]
cite = [(h["epoch"], h["eval_citation_exact"]) for h in HISTORY
        if "eval_citation_exact" in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
if tr:
    ax1.plot(*zip(*tr), marker="o", label="training loss")
if ev:
    ax1.plot(*zip(*ev), marker="s", label="validation loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.grid(alpha=.3)
ax1.set_title("Loss")
if cite:
    ax2.plot(*zip(*cite), marker="o", color="tab:green")
ax2.set_xlabel("epoch"); ax2.set_ylabel("citation exact-match (%)")
ax2.set_title("Citation accuracy (early-stopping criterion)"); ax2.grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, "phase2_5_small_training_curves.png"), dpi=150)
plt.show()

EPOCHS_RUN = max((e for e, _ in ev), default=0)
FINAL_TRAIN_LOSS = tr[-1][1] if tr else float("nan")
BEST_VAL_LOSS = min((v for _, v in ev), default=float("nan"))
BEST_CITATION = max((v for _, v in cite), default=float("nan"))
print(f"epochs run: {EPOCHS_RUN:.0f} | best validation citation accuracy: "
      f"{BEST_CITATION:.1f}%")

---

## Evaluation

Two questions, and they need different setups.

**1. Did the model learn the three behaviours?** Measured on validation, where
the condition of every row is known and controlled. This isolates the
behaviours: use good context, resist bad context, cope without context. It is
the cleanest read, because the condition is assigned rather than whatever
retrieval happened to return.

**2. Does retrieval help end to end?** Measured on the test set, with context
from **live retrieval** — the same retriever, with its real error rate. Three
conditions are run over the identical questions:

| Condition | What it shows |
|---|---|
| `none` | the model with no context — directly comparable to Phase 2 |
| `retrieved` | the realistic system: whatever retrieval actually returns |
| `gold` | oracle context, the ceiling a perfect retriever would reach |

The gap between `none` and `retrieved` is the headline "does retrieval help"
result. The gap between `retrieved` and `gold` is how much is lost to retrieval
error rather than to the model.

In [ ]:
@torch.no_grad()
def generate(rows, batch_size=16, max_new_tokens=MAX_TARGET_LENGTH):
    """Greedy decoding over a list of rows; returns the predicted strings.

    Halves the batch and retries on CUDA OOM rather than losing the run - a
    long decode is the most memory-hungry step in the notebook.
    """
    model.eval()
    preds, start = [], 0
    while start < len(rows):
        chunk = rows[start:start + batch_size]
        try:
            enc = tokenizer([build_prompt(r) for r in chunk],
                            return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_SOURCE_LENGTH).to(model.device)
            # no_repeat_ngram_size stops the degenerate loops greedy
            # decoding falls into when the model is unsure - the Phase 2 run
            # answered one question with "I'm not a witness." thirty times.
            out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 num_beams=1, no_repeat_ngram_size=3)
        except torch.cuda.OutOfMemoryError:
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            torch.cuda.empty_cache()
            print(f"\nOOM - retrying at batch size {batch_size}")
            continue
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        start += len(chunk)
        print(f"\rgenerated {start}/{len(rows)}", end="")
    print()
    return preds

In [ ]:
# score() and summarise() are imported above, from scripts/metrics.py.
# summarise() reports citation metrics as None where the gold answer cites no
# provision at all (transition and scope questions), rather than as a zero.
print("scoring functions ready:", score.__module__, "/", summarise.__module__)

In [ ]:
# ---- 1. validation, by the condition each row was trained under -----------
# The same stratified slice used during training, to keep this step quick.
val_preds = generate(val_sub_rows, batch_size=64)
val_scored = score(val_sub_rows, val_preds,
                   [{"qa_type": r["context_type"]} for r in val_sub_rows])

print("VALIDATION BY CONTEXT CONDITION")
print(f"{'condition':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")
print("-" * 69)
val_by_cond = []
for cond in conds:
    sub = [r for r in val_scored if r["qa_type"] == cond]
    if not sub:
        continue
    s = summarise(sub, cond)
    val_by_cond.append(s)
    print(f"{cond:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"
          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

print("\nHow to read this:")
print("  positive  high   -> the model reads and uses good context")
print("  distractor high  -> it resists bad context (the important one)")
print("  none      decent -> it still works when retrieval returns nothing")
print("  If distractor collapses, the model is trusting context blindly.")

In [ ]:
# ---- 2. the bot's own context, over the test set -------------------------
# This is the change that Phase 3.5 exists for. Evaluation used to paste the
# single top-1 passage; it now calls the same Bot methods that serve a live
# question and that generated the training data, so what is measured here is
# what a user would actually get.
from bot import MAX_CONTEXT_CHUNKS, Bot
from build_context_dataset import (SERVE_K, build_chunk_lookup, correct_chunk)

bot = Bot(model_dir=None, tokenizer=tokenizer)          # assembly half only
by_id, by_doc = build_chunk_lookup(bot.retriever.meta)
print(f"assembling up to {MAX_CONTEXT_CHUNKS} passages per question, "
      f"as bot.py serves them")

test_questions = [r["instruction"] for r in test_rows]
hits = bot.retriever.retrieve_batch(test_questions, k=SERVE_K)

retrieved_ctx, gold_ctx = [], []
gold_available = single_hit = 0
for row, meta, hit in zip(test_rows, test_index, hits):
    chunks = bot.select_chunks(row["instruction"], hits=hit)
    context, used = bot.build_context(chunks)
    retrieved_ctx.append(context)

    gold = correct_chunk(meta["source_chunk_id"], meta["qa_type"], by_id, by_doc)
    used_ids = [c["chunk_id"] for c in used]
    if gold and gold in used_ids:
        gold_available += 1
    if hit and gold and hit[0].chunk_id == gold:
        single_hit += 1

    # The oracle: the same assembly, but with the grounding passage guaranteed
    # present. It bounds what better retrieval alone could buy.
    if gold:
        forced = [c for c in chunks if c["chunk_id"] != gold]
        forced.insert(0, by_id[gold])
        gold_ctx.append(bot.build_context(forced)[0])
    else:
        gold_ctx.append(context)

n = len(test_rows)
print(f"grounding passage present in the assembled context: "
      f"{gold_available}/{n} ({100 * gold_available / n:.1f}%)")
print(f"for comparison, it was the single top-1 hit for: "
      f"{single_hit}/{n} ({100 * single_hit / n:.1f}%)")
gold_hit = single_hit        # kept under its old name for the results file

In [ ]:
CONDITIONS = {
    "none": [""] * len(test_rows),
    "retrieved": retrieved_ctx,
    "gold": gold_ctx,
}

test_results, test_preds_by_cond = {}, {}
for cond, contexts in CONDITIONS.items():
    rows = [{"instruction": r["instruction"], "input": c, "output": r["output"]}
            for r, c in zip(test_rows, contexts)]
    print(f"\n--- generating: {cond} ---")
    preds = generate(rows, batch_size=64)
    test_preds_by_cond[cond] = preds
    test_results[cond] = score(rows, preds, test_index)

In [ ]:
print("=" * 78)
print("HEADLINE: does retrieval help?   (test set, identical questions)")
print("=" * 78)
print(f"{'context':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}")
print("-" * 69)
headline = {}
for cond in ("none", "retrieved", "gold"):
    s = summarise(test_results[cond], cond)
    headline[cond] = s
    print(f"{cond:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"
          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}")

gain = headline["retrieved"]["f1"] - headline["none"]["f1"]
ceiling = headline["gold"]["f1"] - headline["retrieved"]["f1"]
print(f"\n  retrieval gain (retrieved - none): {gain:+.1f} F1")
print(f"  lost to retrieval error (gold - retrieved): {ceiling:+.1f} F1")
print("\n  Compare 'none' against the Phase 2 baseline: that is the cost or")
print("  benefit of context-aware training when no context is supplied.")

In [ ]:
# Per question type, under each condition - where retrieval helps most.
print(f"{'qa_type':<24}" + "".join(f"{c:>14}" for c in
                                   ("none F1", "retrieved F1", "gold F1")))
print("-" * 66)
qa_types = sorted({m["qa_type"] for m in test_index})
by_type_cond = {}
for t in qa_types:
    line = f"{t:<24}"
    by_type_cond[t] = {}
    for cond in ("none", "retrieved", "gold"):
        sub = [r for r in test_results[cond] if r["qa_type"] == t]
        s = summarise(sub, t)
        by_type_cond[t][cond] = s
        line += f"{s['f1']:>13.1f}%"
    print(line)

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(qa_types))
plt.figure(figsize=(11, 4.5))
for i, cond in enumerate(("none", "retrieved", "gold")):
    plt.bar(x + (i - 1) * 0.27,
            [by_type_cond[t][cond]["f1"] for t in qa_types],
            width=0.27, label=cond)
plt.xticks(x, qa_types, rotation=30, ha="right")
plt.ylabel("token F1 (%)"); plt.ylim(0, 100)
plt.title("Does retrieved context help? By question type")
plt.legend(); plt.grid(axis="y", alpha=.3); plt.tight_layout()
plt.savefig("/content/retrieval_gain_by_qa_type.png", dpi=150)
plt.show()

In [ ]:
# ---- confusion set: does the bot's context now change the answers? -------
# Three conditions on the same 44 questions. "single passage" reproduces what
# Phase 2.5 and Phase 3 were measured with, so the comparison is exact.
conf_questions = [e["instruction"] for e in confusion]
conf_hits = bot.retriever.retrieve_batch(conf_questions, k=SERVE_K)

single_ctx, bot_ctx = [], []
for q, hit in zip(conf_questions, conf_hits):
    top = [bot.retriever.meta[bot.row_of_chunk[hit[0].chunk_id]]] if hit else []
    single_ctx.append(bot.build_context(top, max_chunks=1)[0] if top else "")
    bot_ctx.append(bot.build_context(bot.select_chunks(q, hits=hit))[0])

conf_metas = [{"qa_type": e["mapping_type"]} for e in confusion]
conf_results = {}
for cond, contexts in (("none", [""] * len(confusion)),
                       ("single passage", single_ctx),
                       ("bot context", bot_ctx)):
    rows = [{"instruction": e["instruction"], "input": c, "output": e["output"]}
            for e, c in zip(confusion, contexts)]
    preds = generate(rows, batch_size=64)
    conf_results[cond] = (preds, score(rows, preds, conf_metas))

print("=" * 78)
print("CONFUSION SET - the old-vs-new questions held out of every split")
print("=" * 78)
print(f"{'context':<26}{'n':>6}{'EM':>9}{'F1':>9}{'citeF1':>9}{'allCites':>10}"
      f"{'exact':>9}")
print("-" * 78)
conf_summary = {}
for cond in ("none", "single passage", "bot context"):
    s = summarise(conf_results[cond][1], cond)
    conf_summary[cond] = s
    print(f"{cond:<26}{s['n']:>6}{pct(s['exact_match'])}{pct(s['f1'])}"
          f"{pct(s['citation_f1'])}{pct(s['all_citations_present'])}"
          f"{pct(s['citation_exact'])}")

print("\nallCites only asks whether the right provisions are PRESENT, so an answer")
print("naming the right section and inventing a second one still passes. 'exact'")
print("requires the citation sets to match - read that column for the removed and")
print("no-counterpart questions, where the correct answer is 'there is none'.")
print("\nPhase 4 puts the same questions to a general-purpose LLM for a fourth row.")

# Per kind, which is where the Phase 4 comparison is actually decided: a
# general-purpose LLM does well on collisions and repeals, which are widely
# written about, and badly on merged families, where naming every constituent
# old section requires the concordance rather than general knowledge.
print(f"\n{'kind':<14}{'n':>4}" + "".join(f"{c:>17}" for c in
      ("none", "single passage", "bot context")))
print("-" * 69)
conf_by_kind = {}
for kind in sorted({m["qa_type"] for m in conf_metas}):
    row = {}
    line = f"{kind:<14}"
    n = 0
    for cond in ("none", "single passage", "bot context"):
        sub = [r for r in conf_results[cond][1] if r["qa_type"] == kind]
        s = summarise(sub, kind)
        row[cond] = s
        n = s["n"]
        line += f"{pct(s['all_citations_present']):>17}"
    conf_by_kind[kind] = row
    print(f"{kind:<14}{n:>4}" + line[14:])

In [ ]:
# The qualitative view, side by side.
for kind in ("collision", "merged", "split", "removed"):
    i = next((i for i, e in enumerate(confusion)
              if e["mapping_type"] == kind), None)
    if i is None:
        continue
    e = confusion[i]
    print("=" * 78)
    print(f"[{kind}]")
    print("  Q         :", e["instruction"])
    print("  gold      :", textwrap.shorten(e["output"], 200, placeholder=" ..."))
    for cond in ("none", "retrieved"):
        pred = conf_results[cond][0][i]
        ok = extract_refs(e["output"]) <= extract_refs(pred)
        print(f"  {cond:<10}: [{'OK ' if ok else 'MISS'}] "
              + textwrap.shorten(pred, 190, placeholder=" ..."))

---

## Saving the model

In [ ]:
# The model is already on Drive - it was saved the moment training finished.
# This only publishes it to the Hugging Face Hub, if a token is set.
MODEL_ARTIFACT = FINAL_DIR

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    from huggingface_hub import HfApi
    who = HfApi(token=hf_token).whoami()["name"]
    repo_id = f"{who}/{HF_REPO_ID}"
    model.push_to_hub(repo_id, token=hf_token)
    tokenizer.push_to_hub(repo_id, token=hf_token)
    MODEL_ARTIFACT = f"https://huggingface.co/{repo_id}"
    print("published:", MODEL_ARTIFACT)
else:
    print("Model saved on Google Drive at", FINAL_DIR)
    print("(Add an HF_TOKEN secret and re-run this cell to publish it.)")

In [ ]:
results = {
    "run": "phase3.6 context-aware + negation shapes (light: flan-t5-small, resumable)",
    "model": MODEL_NAME,
    "epochs_run": float(EPOCHS_RUN),
    "best_val_loss": float(BEST_VAL_LOSS),
    "best_val_citation_exact": float(BEST_CITATION),
    "training_minutes": round(TRAIN_MINUTES, 1),
    "train_rows": len(ctx["train"]),
    "context_mix": dict(Counter(r["context_type"] for r in ctx["train"])),
    "validation_by_context_type": val_by_cond,
    "test_headline": headline,
    "test_by_qa_type_and_condition": by_type_cond,
    "retrieval_top1_accuracy": round(100 * gold_hit / len(test_rows), 1),
    "grounding_passage_in_context": round(100 * gold_available / len(test_rows), 1),
    "max_context_chunks": MAX_CONTEXT_CHUNKS,
    "confusion": conf_summary,
    "confusion_by_kind": conf_by_kind,
    "confusion_predictions": {
        cond: conf_results[cond][0] for cond in conf_results},
    "model_artifact": MODEL_ARTIFACT,
}
dest = os.path.join(DRIVE_DIR, "phase3_6_results.json")
os.makedirs(os.path.dirname(dest), exist_ok=True)
with open(dest, "w", encoding="utf-8") as fh:
    json.dump(results, fh, indent=2)
print("written to", dest)
print(f"\nretrieval gain on test: {gain:+.1f} F1  "
      f"(none {headline['none']['f1']:.1f} -> retrieved "
      f"{headline['retrieved']['f1']:.1f}, gold {headline['gold']['f1']:.1f})")

---

## Reading the results

**The validation-by-condition table** says whether the three behaviours were
learnt. `positive` high and `distractor` collapsing means the model has learnt
to trust context rather than to read it — which would be worse than Phase 2,
because a confident wrong answer sourced from a retrieved passage looks
authoritative.

**The headline table** is the paper's central result. Three numbers over the
same questions: no context, live retrieval, oracle context. The first gap is
what retrieval buys; the second is what better retrieval could still buy.

**Compare `none` here against Phase 2's overall F1.** They are the same
questions with the same prompt and no context in either case, so the difference
is purely the effect of context-aware training on unaided performance. A small
drop is an acceptable trade; a large one means the model has become dependent
on context.

**The confusion set** is the differentiator thesis. Phase 4 adds a third
column — the same questions put to a general-purpose LLM.

### What Phase 3 adds

The retriever and this model become a single callable bot: retrieve, build the
prompt, generate, and cite the `source_url` of the chunk used. The pieces all
exist — `scripts/retrieve.py` is the same module this notebook imports.